In [1]:
import sys
sys.path.append("..")
import os
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

import torch
import torch.nn.functional as F
import numpy as np
import yaml
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from tqdm import tqdm
import h5py

from copy import deepcopy

from src.models.base_models import create_model
from src.models.domain_discriminator import DomainDiscriminator, create_da_model
from src.data.hcut_numu_dataset import HCutNuMuDataset, create_hcut_numu_dataloader, create_hcut_numu_dataloader_from_ds

In [2]:
test_dataset = HCutNuMuDataset(
    h5_path="/net/62/home3/ivkhar/Baikal/data/h5s/baikal_mc_merged.h5",
    events_per_particle={'muatm_2020': 100_000, 'nuatm_2020': 100_000, 'nue2_2020': 100_000},  # Test portion
    particle_types=['muatm_2020', 'nue2_2020', 'nuatm_2020'],
    neutrino_types=['nue2_2020', 'nuatm_2020'],
    max_hits=500,
    sampling_config={
        'muatm_2020': {'mode': 'range', 'start_event': 100_000},
        'nue2_2020': {'mode': 'range', 'start_event': 0},
        'nuatm_2020': {'mode': 'range', 'start_event': 0}
        },
    device='cuda:0',
    shuffle_events=True,
    balance_classes=True
)
test_dataset._get_stats()

2026-02-06 07:51:52,091 [INFO] Lazy range loading for muatm_2020: 100000 events loaded (range 100000-200000)
2026-02-06 07:51:58,054 [INFO] Loaded 100000 events for muatm_2020
2026-02-06 07:51:58,067 [INFO] Loaded 100000 events from muatm_2020 (neutrino_type=False)
2026-02-06 07:51:58,068 [INFO]   -> Signal (1): 0, Background (0): 100000
2026-02-06 07:51:58,132 [INFO] Lazy range loading for nue2_2020: 100000 events loaded (range 0-100000)
2026-02-06 07:52:03,833 [INFO] Loaded 100000 events for nue2_2020
2026-02-06 07:52:03,866 [INFO] Loaded 100000 events from nue2_2020 (neutrino_type=True)
2026-02-06 07:52:03,867 [INFO]   -> Signal (1): 78722, Background (0): 21278
2026-02-06 07:52:03,873 [INFO]   -> Average signal hits per event: 11.5
2026-02-06 07:52:03,932 [INFO] Lazy range loading for nuatm_2020: 100000 events loaded (range 0-100000)
2026-02-06 07:52:09,274 [INFO] Loaded 100000 events for nuatm_2020
2026-02-06 07:52:09,294 [INFO] Loaded 100000 events from nuatm_2020 (neutrino_type=

{'total_events': 130075,
 'signal_events': 65038,
 'background_events': 65037,
 'class_balance': 0.5000038439361907,
 'h_min_threshold': 5,
 'total_hits': 8414961,
 'min_hits': 29,
 'max_hits': 331,
 'mean_hits': 64.69314575195312,
 'std_hits': 12.488245964050293,
 'min_signal_hits': 0,
 'max_signal_hits': 273,
 'mean_signal_hits': np.float64(8.338750720738036),
 'std_signal_hits': np.float64(8.79326993841417),
 'feature_means': [3.157273769378662,
  -1.8730227679952804e-07,
  -0.48858290910720825,
  2.6592705249786377,
  59.19394302368164],
 'feature_stds': [102.34027862548828,
  1343.8521728515625,
  39.90843963623047,
  38.675167083740234,
  150.4605712890625],
 'feature_names': ['amplitude', 'time', 'x', 'y', 'z']}

In [3]:
len(test_dataset)

130075

In [12]:
(np.array(test_dataset.signal_hit_counts)>=5).sum()

np.int64(46216)

In [28]:
MUONS_HGE5_PART = 0.46
SIGNAL_PART_NUATM = 0.325
SIGNAL_PART_NUE2 = 0.787

# Constraints: 
# 1) signal nuatm == signal nue2
# - 2) muons with h>=5 == signal nuatm + signal nue2 # NOT RESOLVABLE!!!
# 3) Background == Signal
# 4) Sum == 100_000

N_TO_LOAD = 1_000_000

n_nuatm = N_TO_LOAD / 4 / SIGNAL_PART_NUATM
n_nunu2 = N_TO_LOAD / 4 / SIGNAL_PART_NUE2
n_muatm = N_TO_LOAD - n_nuatm - n_nunu2

In [ ]:
MUONS_HGE5_PART*n_muatm, n_nuatm * SIGNAL_PART_NUATM + n_nunu2*SIGNAL_PART_NUE2

(-86892.7768546574, 500000.0)

In [32]:
n_nuatm, n_nunu2

(769230.7692307692, 317662.00762388814)

In [20]:
test_dataset = HCutNuMuDataset(
    h5_path="/net/62/home3/ivkhar/Baikal/data/h5s/baikal_mc_merged.h5",
    events_per_particle={'muatm_2020': n_muatm, 'nuatm_2020': n_nuatm, 'nue2_2020': n_nunu2},  # Test portion
    particle_types=['muatm_2020', 'nue2_2020', 'nuatm_2020'],
    neutrino_types=['nue2_2020', 'nuatm_2020'],
    max_hits=500,
    sampling_config={
        'muatm_2020': {'mode': 'range', 'start_event': 0},
        'nue2_2020': {'mode': 'range', 'start_event': 0},
        'nuatm_2020': {'mode': 'range', 'start_event': 0}
        },
    device='cuda:0',
    shuffle_events=True
)
test_dataset._get_stats()

{'total_events': 108689,
 'signal_events': 49877,
 'background_events': 58812,
 'class_balance': 0.45889648446484926,
 'h_min_threshold': 5,
 'total_hits': 6800514,
 'min_hits': 29,
 'max_hits': 364,
 'mean_hits': 62.56855773925781,
 'std_hits': 11.712785720825195,
 'min_signal_hits': 0,
 'max_signal_hits': 319,
 'mean_signal_hits': np.float64(6.1282466486949),
 'std_signal_hits': np.float64(7.592099589060259),
 'feature_means': [2.857778310775757,
  6.540589225778604e-08,
  -0.5877465009689331,
  2.7518692016601562,
  58.90861129760742],
 'feature_stds': [101.62625885009766,
  1363.588623046875,
  39.9561767578125,
  38.69140625,
  151.13880920410156],
 'feature_names': ['amplitude', 'time', 'x', 'y', 'z']}

In [19]:
N_ATM, N_MU

(44964, 10071)

In [13]:
test_dataset = HCutNuMuDataset(
    h5_path="/net/62/home3/ivkhar/Baikal/data/h5s/baikal_mc_merged.h5",
    events_per_particle={'muatm_2020': 0, 'nuatm_2020': 100_000, 'nue2_2020': 0},  # Test portion
    particle_types=['muatm_2020', 'nue2_2020', 'nuatm_2020'],
    neutrino_types=['nue2_2020', 'nuatm_2020'],
    max_hits=500,
    sampling_config={
        'muatm_2020': {'mode': 'range', 'start_event': 0},
        'nue2_2020': {'mode': 'range', 'start_event': 0},
        'nuatm_2020': {'mode': 'range', 'start_event': 0}
        },
    device='cuda:0',
    shuffle_events=True
)
test_dataset._get_stats()

{'total_events': 100000,
 'signal_events': 32519,
 'background_events': 67481,
 'class_balance': 0.32519,
 'h_min_threshold': 5,
 'total_hits': 6049553,
 'min_hits': 29,
 'max_hits': 113,
 'mean_hits': 60.49552917480469,
 'std_hits': 9.227884292602539,
 'min_signal_hits': 0,
 'max_signal_hits': 37,
 'mean_signal_hits': np.float64(4.06025),
 'std_signal_hits': np.float64(2.4008998182972983),
 'feature_means': [1.2915380001068115,
  -1.2107040925357637e-09,
  -0.6366331577301025,
  2.8332226276397705,
  60.05900192260742],
 'feature_stds': [7.361569404602051,
  1383.95703125,
  39.98104476928711,
  38.692989349365234,
  150.92971801757812],
 'feature_names': ['amplitude', 'time', 'x', 'y', 'z']}

In [14]:
test_dataset = HCutNuMuDataset(
    h5_path="/net/62/home3/ivkhar/Baikal/data/h5s/baikal_mc_merged.h5",
    events_per_particle={'muatm_2020': 0, 'nuatm_2020': 0, 'nue2_2020': 100_000},  # Test portion
    particle_types=['muatm_2020', 'nue2_2020', 'nuatm_2020'],
    neutrino_types=['nue2_2020', 'nuatm_2020'],
    max_hits=500,
    sampling_config={
        'muatm_2020': {'mode': 'range', 'start_event': 0},
        'nue2_2020': {'mode': 'range', 'start_event': 0},
        'nuatm_2020': {'mode': 'range', 'start_event': 0}
        },
    device='cuda:0',
    shuffle_events=True
)
test_dataset._get_stats()

{'total_events': 100000,
 'signal_events': 78722,
 'background_events': 21278,
 'class_balance': 0.78722,
 'h_min_threshold': 5,
 'total_hits': 6788237,
 'min_hits': 31,
 'max_hits': 364,
 'mean_hits': 67.88236999511719,
 'std_hits': 15.659940719604492,
 'min_signal_hits': 0,
 'max_signal_hits': 319,
 'mean_signal_hits': np.float64(11.5257),
 'std_signal_hits': np.float64(12.889186146145923),
 'feature_means': [6.571157932281494,
  -1.0119271820485665e-07,
  -0.45846331119537354,
  2.617192506790161,
  56.215667724609375],
 'feature_stds': [173.58973693847656,
  1315.92822265625,
  39.89316177368164,
  38.67970275878906,
  151.38929748535156],
 'feature_names': ['amplitude', 'time', 'x', 'y', 'z']}

In [15]:
100000* 0.787/(1.787)

44040.29099048685

In [5]:
dl = create_hcut_numu_dataloader_from_ds(test_dataset, 32, True, None, shuffle_batch=False, augmentation_config=None)

for batch in dl:
    print(batch['labels'])
    break

tensor([False, False, False, False,  True, False,  True, False,  True,  True,
         True, False, False, False, False,  True, False, False, False, False,
        False, False,  True,  True, False,  True, False, False, False,  True,
         True, False], device='cuda:0')


In [7]:
with h5py.File(test_dataset.h5_path, 'r') as f:
    events, labels, magic_numbers, hit_counts, event_ids, channel_ids, signal_hit_counts = test_dataset._load_particle_data(f, test_dataset.particle_types[0], False, 50_000)

In [17]:
len(hit_counts)

50000

In [16]:
len(signal_hit_counts)

50000

In [9]:
len(signal_hit_counts), len(labels)

(50000, 50000)

In [5]:
test_dataset.particle_types

['muatm_2020', 'nue2_2020', 'nuatm_2020']